# 06 — Data sources

Data sources (databases, historians, file paths, camera streams, ...) are
registered in a three-tier Redis convention, each tier a hash keyed by
source name, value a JSON-encoded record:

| Key | Scope |
|---|---|
| `data-sources:global` | visible to every worker and node |
| `data-sources:worker:{APP_ID}` | scoped to one worker *type* |
| `data-sources:local:{NODE_ADDRESS}` | scoped to a single physical node |

There's no dedicated SDK class for this - it's a direct Redis hash
convention, the same one the Composer UI's Data Sources page reads and
writes.

In [ ]:
import os, json, time
import pandas as pd
from scarlets.utils.ScarletUtils import redisConnect

os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")

r = redisConnect(decode_responses=True)

def source_key(tier, qualifier=None):
    return f"data-sources:{tier}:{qualifier}" if qualifier else f"data-sources:{tier}"

def register(tier, name, qualifier=None, **fields):
    entry = {"name": name, "registered_at": time.time(), **fields}
    r.hset(source_key(tier, qualifier), name, json.dumps(entry))

def read(tier, name, qualifier=None):
    raw = r.hget(source_key(tier, qualifier), name)
    return json.loads(raw) if raw else None

## Cleanup

In [ ]:
def cleanup():
    patterns = ["data-sources:global", "data-sources:worker:*", "data-sources:local:*"]
    for pattern in patterns:
        keys = list(r.scan_iter(match=pattern))
        if keys:
            r.delete(*keys)

cleanup()
print("cleaned up")

## Global tier

Visible to every worker and every node - for shared infrastructure like a
plant historian every agent might want to query.

In [ ]:
register("global", "plant-historian", type="PI", uri="pi://10.0.0.50",
         description="OSIsoft PI historian")

read("global", "plant-historian")

## Worker tier — scoped by APP_ID

Two different worker *types* (`anomaly_detector`, `forecaster`) each get
their own isolated namespace, even if they happen to register a source
with the same name.

In [ ]:
register("worker", "local-model", qualifier="anomaly_detector",
         type="file", uri="/models/anomaly_v1.pkl")
register("worker", "local-model", qualifier="forecaster",
         type="file", uri="/models/forecaster_v3.pkl")

print("anomaly_detector sees:", read("worker", "local-model", "anomaly_detector"))
print("forecaster sees:      ", read("worker", "local-model", "forecaster"))

## Local tier — scoped by node

A source tied to one physical machine - an attached camera, a local sensor
- registered only for that node's address.

In [ ]:
register("local", "edge-camera", qualifier="osu-node-1",
         type="RTSP", uri="rtsp://10.0.1.5/stream")

read("local", "edge-camera", "osu-node-1")

## Listing everything across all three tiers

A quick inventory - every registered source, tagged with which tier and
scope it belongs to.

In [ ]:
rows = []
for tier, pattern in [("global", "data-sources:global"),
                       ("worker", "data-sources:worker:*"),
                       ("local", "data-sources:local:*")]:
    for key in r.scan_iter(match=pattern):
        qualifier = key.split(":", 2)[2] if tier != "global" else None
        for name, raw in r.hgetall(key).items():
            entry = json.loads(raw)
            rows.append({"tier": tier, "scope": qualifier, "name": name,
                         "type": entry.get("type"), "uri": entry.get("uri")})

pd.DataFrame(rows)

## Cleanup (teardown)

In [ ]:
cleanup()
print("cleaned up")